In [3]:
from anthropic import Anthropic

In [4]:
client = Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000, 
    messages=[
        {"role": "user", "content": "Hello there! Give me a motivational quote for the day. Don't ask any follow up questions or any preambles"}
    ]
)

In [5]:
print(response.content[0].text)

"The only way to do great work is to love what you do." — Steve Jobs


## Simple single-turn messaging with Claude

In [7]:
def translate(word, language):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"Translate the following word from English to {language.capitalize()}\n<word>{word}</word>",
            }
        ],
    )
    return response.content[0].text

print(translate("Morning", "Spanish"))
print(translate("Morning", "Telugu"))

The translation of the word **"Morning"** from English to Spanish is:

**"Mañana"**
The Telugu translation of **"Morning"** is:

# **ఉదయం**
*(Udayam)*


## Putting words in Claude's mouth

In [8]:
response = client.messages.create(
    model = "claude-haiku-4-5",
    max_tokens=500,
    messages = [
        {"role": "user", "content": "Please tell me Haiku"},
        {"role": "assistant", "content": "crystal white mountains"}
    ]
)

print("crystal white mountains" + response.content[0].text)

crystal white mountains
snow falls on silent pine trees
winter's quiet song


## Few-shot prompting

In [11]:
import json

few_shot_conversation = [
    {
        "role": "user",
        "content": "The smart girl solved the hard math puzzle very quickly.",
    },
    {
        "role": "assistant",
        "content": json.dumps(
            [
                {"word": "The", "tag": "DET"},
                {"word": "smart", "tag": "ADJ"},
                {"word": "girl", "tag": "NOUN"},
                {"word": "solved", "tag": "VERB"},
                {"word": "the", "tag": "DET"},
                {"word": "hard", "tag": "ADJ"},
                {"word": "math", "tag": "NOUN"},
                {"word": "puzzle", "tag": "NOUN"},
                {"word": "very", "tag": "ADV"},
                {"word": "quickly", "tag": "ADV"},
            ]
        ),
    },
    {
        "role": "user",
        "content": "My old friend lives in a beautiful house near the river.",
    },
    {
        "role": "assistant",
        "content": json.dumps(
            [
                {"word": "My", "tag": "PRON"},
                {"word": "old", "tag": "ADJ"},
                {"word": "friend", "tag": "NOUN"},
                {"word": "lives", "tag": "VERB"},
                {"word": "in", "tag": "ADP"},
                {"word": "a", "tag": "DET"},
                {"word": "beautiful", "tag": "ADJ"},
                {"word": "house", "tag": "NOUN"},
                {"word": "near", "tag": "ADP"},
                {"word": "the", "tag": "DET"},
                {"word": "river", "tag": "NOUN"},
            ]
        ),
    },
    {
        "role": "user",
        "content": "We saw three black birds fly across the bright blue sky.",
    },
    {
        "role": "assistant",
        "content": json.dumps(
            [
                {"word": "We", "tag": "PRON"},
                {"word": "saw", "tag": "VERB"},
                {"word": "three", "tag": "NUM"},
                {"word": "black", "tag": "ADJ"},
                {"word": "birds", "tag": "NOUN"},
                {"word": "fly", "tag": "VERB"},
                {"word": "across", "tag": "ADP"},
                {"word": "the", "tag": "DET"},
                {"word": "bright", "tag": "ADJ"},
                {"word": "blue", "tag": "ADJ"},
                {"word": "sky", "tag": "NOUN"},
            ]
        ),
    },
    {
        "role": "user",
        "content": "The children ate sweet apples and drank cold milk for lunch."
    }
]

response = client.messages.create(
    model = "claude-haiku-4-5",
    max_tokens=500,
    messages = few_shot_conversation
) 

json.loads(response.content[0].text)

[{'word': 'The', 'tag': 'DET'},
 {'word': 'children', 'tag': 'NOUN'},
 {'word': 'ate', 'tag': 'VERB'},
 {'word': 'sweet', 'tag': 'ADJ'},
 {'word': 'apples', 'tag': 'NOUN'},
 {'word': 'and', 'tag': 'CCONJ'},
 {'word': 'drank', 'tag': 'VERB'},
 {'word': 'cold', 'tag': 'ADJ'},
 {'word': 'milk', 'tag': 'NOUN'},
 {'word': 'for', 'tag': 'ADP'},
 {'word': 'lunch', 'tag': 'NOUN'}]

## Multi-turn chat with Claude

In [22]:
def chatbot():
    print("==========Talk to Claude==========")
    print("(Send `Bye` to end chat!)")
    print("==================================\n")
    conversation = []
    while True:
        user_msg = input()
        print(f"You: {user_msg}")
        print("-"*20)
        if user_msg.strip() == "Bye":
            break
        conversation.append({"role": "user", "content": user_msg})
        response = client.messages.create(
            model = "claude-sonnet-4-6",
            max_tokens=1000,
            messages = conversation,
        )
        assistant_msg = response.content[0].text
        print(f"Assistant: {assistant_msg}")
        print("-"*20)
        conversation.append({"role": "assistant", "content": assistant_msg})
chatbot()

==========Talk to Claude==========
(Send `Bye` to end chat!)

You: Hey there! How are you doing?
--------------------
Assistant: Hey! I'm doing well, thanks for asking. I'm ready to help with whatever you've got on your mind. How are you doing?
--------------------
You: I'm doing well! Thanks for asking. Can you tell me one good place to visit near Houston in June?
--------------------
Assistant: Great to hear you're doing well!

One great place to visit near Houston in June is **Galveston Island**. It's only about an hour's drive from Houston and offers:

- **Beautiful beaches** along the Gulf Coast
- **Historic downtown** with shops and restaurants
- **Moody Gardens** with its aquarium and attractions
- **The Strand** historic district

Just keep in mind that June can be **hot and humid**, so be sure to stay hydrated and use sunscreen. It's also worth checking weather forecasts since it's hurricane season. But overall it's a popular and fun destination, especially if you enjoy the be

## Parameters (Temperature, max_tokens, stop_sequence, system)

In [10]:
def generate_questions(topic, num_questions):
    system_prompt = f"""
        You are an expert on `{topic}`. You raise thought-provoking questions on this topic.
        Style rules:
        - Just generate the questions as a numbered list
        - Don't insert any preambles in your reponse
        - Don't ask any follow-up questions to the user at the end.
    """
    user_prompt = f"Generate {num_questions} questions on the topic: {topic}"
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1000,
        system=system_prompt,
        stop_sequences=[f"{num_questions+1}. "],
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )
    return response

In [11]:
response = generate_questions("Elephant Seals", 4)

print(response.content[0].text)
print(response.stop_reason)

1. Given that male elephant seals can fast for up to 120 days during breeding season, what physiological mechanisms allow them to survive such extreme periods without food or water?

2. How might climate change and shifting ocean currents affect the food availability and migration patterns of elephant seals in their traditional feeding grounds?

3. What evolutionary advantages could explain why male elephant seals developed their distinctive inflatable proboscis, and how does this feature influence their reproductive success?

4. Since elephant seal populations were hunted to near extinction in the 19th century and have since recovered, what genetic consequences might this population bottleneck have had on their species' long-term adaptability?
end_turn


## Streaming Responses

In [22]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    messages=[{"role": "user", "content": "Give me a paragraph about Chicago. No preamble."}],
    stream=True,
)

for event in response:
    if event.type == "message_start":
        print(f"Input Tokens: {event.message.usage.input_tokens}")
    elif event.type == "message_delta":
        print(f"\nOutput Tokens: {event.usage.output_tokens}")
    elif event.type == "content_block_delta":
        print(event.delta.text, end="\n", flush=True)    

Input Tokens: 19
Chicago, the third
-largest city in the United States, sits on the shores of Lake Michigan in Illinois and stands as a
 global hub for commerce, culture, and architecture. Known as the "Windy City," Chicago has a
 rich history of innovation and influence, from its role as a major transportation and industrial
 center to its reputation as the birthplace of the skyscraper. The city boasts world-class museums like the Art Institute and Field Museum, stunning architecture ranging
 from historic Victorian homes to modern glass towers, and a vibrant cultural scene featuring jazz
, blues, and diverse neighborhoods. Home to iconic landmarks such as the Willis Tower, Millennium Park, and Navy Pier, Chicago attra
cts millions of visitors annually while remaining a dynamic place where diverse communities, ambitious
 professionals, and creative minds continue to shape its evolving character.

Output Tokens: 173


In [25]:
from anthropic import AsyncAnthropic

client = AsyncAnthropic()

async def streaming_with_helpers():
    async with client.messages.stream(
            model="claude-haiku-4-5",
            max_tokens=500,
            messages=[
                {"role": "user", "content": "Generate a 5 sentence haiku about serenity. No preamble."}
            ]
        ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)
        
        final_message = await stream.get_final_message() 
    print("\n\nSTREAMING IS DONE. FINAL MESSAGE BELOW\n ")
    print(final_message.to_json())

await streaming_with_helpers()

# Serenity

Still water reflects
The moon's gentle silver glow—
Peace within, without.

Breath flows like soft wind
Through valleys of quiet mind—
Calm blooms like lotus.

Silence speaks volumes,
Each moment a gentle gift—
Being is enough.

STREAMING IS DONE. FINAL MESSAGE BELOW
 
{
  "id": "msg_01DoNNABJiDV4SU9dY5dd1DH",
  "content": [
    {
      "citations": null,
      "text": "# Serenity\n\nStill water reflects\nThe moon's gentle silver glow—\nPeace within, without.\n\nBreath flows like soft wind\nThrough valleys of quiet mind—\nCalm blooms like lotus.\n\nSilence speaks volumes,\nEach moment a gentle gift—\nBeing is enough.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache

## Chat with Streaming 

In [5]:
import asyncio
from anthropic import AsyncAnthropic

client = AsyncAnthropic()

# ANSI color codes
BLUE = "\033[94m"
GREEN = "\033[92m"
RESET = "\033[0m"

async def chat_with_claude():
    conversation = []
    print("=====================CHAT WITH CLAUDE=====================")
    print("Send `Bye` to end chat!")
    print("==========================================================")
    while True:
        user_msg = input()
        print(f"{BLUE}YOU: {RESET}{user_msg}", flush=True)
        if user_msg == "Bye":
            break
        conversation.append({"role": "user", "content": user_msg})            
        async with client.messages.stream(
                model="claude-haiku-4-5",
                max_tokens=500,
                messages=conversation
            ) as stream:
            print(f"{GREEN}CLAUDE: {RESET}", end="", flush=True)
            async for text in stream.text_stream:
                print(f"{GREEN}{text}{RESET}", end="", flush=True)
            final_text = await stream.get_final_text()
        conversation.append({"role": "assistant", "content": final_text})
        print()

await chat_with_claude()
        

=====================CHAT WITH CLAUDE=====================
Send `Bye` to end chat!
YOU: Can you explain Karma Yoga from Bhagavad Gita in one sentence?
CLAUDE: # Karma Yoga

Karma Yoga is the yoga of selfless action—performing your duties without attachment to the results, dedicating your work to the divine rather than personal gain.
YOU: Elaborate on it please
CLAUDE: # Karma Yoga: Detailed Explanation

## Core Principle
Karma Yoga teaches that you should perform your duties (dharma) wholeheartedly, but remain detached from the outcomes. Your responsibility is the action itself, not its fruits.

## Key Concepts

**Selfless Action**
- Work without ego or personal ambition
- Perform actions as service rather than for reward
- Dedicate results to God or a higher purpose

**Detachment from Results**
- You cannot control outcomes, only your effort
- Success and failure are treated equally
- This detachment reduces anxiety and suffering

**Duty (Dharma)**
- Everyone has specific duties based

## Images in prompt

In [10]:
import base64
import mimetypes
from anthropic import Anthropic

img_paths = [
    "./assets/page1.png",
    "./assets/page2.png",
    "./assets/page3.png",
    "./assets/page4.png",
    "./assets/page5.png",
]

client = Anthropic()


# Helper function
def create_img_msg(img_path):
    with open(img_path, "rb") as f:
        img_bytes = f.read()

        # Encode to Base64
        img_b64 = base64.b64encode(img_bytes)

        # Serialize Base64 data to a String
        img_b64_str = img_b64.decode(encoding="utf-8")

    img_mimetype, _ = mimetypes.guess_type(img_path)
    return {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": img_mimetype,
            "data": img_b64_str,
        },
    }


def generate_transcript(img_path):
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=5000,
        messages=[
            {
                "role": "user",
                "content": [
                    create_img_msg(img_path),
                    {
                        "type": "text",
                        "text": "Transcribe the text from this page of a research paper as accurately as possible",
                    },
                ],
            }
        ],
    )
    return response.content[0].text

generate_transcript(img_paths[0])

'# Many-shot Jailbreaking\n\n**Cem Anil**<sup>123</sup> **Esin Durmus**<sup>1</sup> **Mrinank Sharma**<sup>1</sup> **Joe Benton**<sup>1</sup> **Sandipan Kundu**<sup>1</sup> **Joshua Batson**<sup>1</sup> **Nina Rimsky**<sup>1</sup> **Meg Tong**<sup>1</sup> **Jesse Mu**<sup>1</sup> **Daniel Ford**<sup>1</sup> **Francesco Mosconi**<sup>1</sup> **Rajashree Agrawal**<sup>∗</sup> **Rylan Schaeffer**<sup>∗4</sup> **Naomi Bashkansky**<sup>46</sup> **Samuel Svenningsen**<sup>4</sup> **Mike Lambert**<sup>1</sup> **Ansh Radhakrishnan**<sup>1</sup> **Carson Denison**<sup>1</sup> **Evan J Hubinger**<sup>1</sup> **Yuntao Bai**<sup>1</sup> **Trenton Bricken**<sup>1</sup> **Timothy Maxwell**<sup>1</sup> **Nicholas Schiefer**<sup>1</sup> **Jamie Sully**<sup>1</sup> **Alex Tamkin**<sup>1</sup> **Tamera Lanham**<sup>1</sup> **Karina Nguyen**<sup>1</sup> **Tomasz Korbak**<sup>1</sup>\n\n**Jared Kaplan**<sup>1</sup> **Deep Ganguli**<sup>1</sup> **Samuel R. Bowman**<sup>1</sup> **Ethan Perez**<sup>∗1</sup> 

In [12]:
def summarize():
    paper_transcript = ""
    for img_path in img_paths:
        print(f"Generating transcript for `{img_path}`...")
        paper_transcript += generate_transcript(img_path)
        print(f"Transcript generated!")

    print("\n==========================Paper Summary==========================\n")

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=5000,
        messages=[
            {
                "role": "user",
                "content": f"""
                    Summarize the research paper and explain it in plain non-technical terms.
                    Target audience: non-research readers.
                    Explain in 3 paragraphs 
                    Avoid using technical argons or abbreviations.
                    Use analogies wherever possible.
                    <paper>
                    {paper_transcript}
                    </paper>
                """,
            }
        ],
    )
    
    print(response.content[0].text)
    
summarize()

Generating transcript for `./assets/page1.png`...
Transcript generated!
Generating transcript for `./assets/page2.png`...
Transcript generated!
Generating transcript for `./assets/page3.png`...
Transcript generated!
Generating transcript for `./assets/page4.png`...
Transcript generated!
Generating transcript for `./assets/page5.png`...
Transcript generated!

==========================Paper Summary==========================

# Many-Shot Jailbreaking: Understanding a New Vulnerability in AI Chatbots

This research reveals a surprisingly simple way to trick AI chatbots into behaving badly by exploiting their newly expanded "memory." Modern AI assistants like ChatGPT and Claude can now read and remember much longer conversations—equivalent to several novels worth of text. The researchers discovered that by showing these AI systems hundreds of examples of harmful behavior in a fake conversation, they could convince the AI to adopt that same harmful behavior. Think of it like peer pressure: 